# 05 — Diarize the episodes (who spoke when)

Runs pyannote `speaker-diarization-community-1` over every episode's full audio and writes one
`diarization.json` to Drive (`MyDrive/nepanglish-asr/diarization/`). The harness imports it with

```bash
backend/.venv/bin/python scripts/import_diarization.py diarization.json
```

which feeds the speaker colours on the karaoke line (D78). The harness never runs this model itself (D58):
it runs here, on a GPU, whenever there are new episodes or a better diarizer. An A100 does all 42 episodes
(24 h) in about 17 minutes; each episode is saved as it finishes, so a disconnect resumes where it stopped.

**Before the first run:** accept the model's terms at huggingface.co/pyannote/speaker-diarization-community-1
with the account whose `HF_TOKEN` is in the Colab secrets, and pick a GPU runtime.

**Run the Setup cell by hand** (the Drive consent popup needs the Colab UI). If it fails with a numpy
`ImportError`, pip replaced numpy under the running kernel: *Runtime → Restart session* and run it again.

In [ ]:
import subprocess
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout or "NO GPU")

## Setup

In [ ]:
%pip install -q "pyannote.audio==4.0.7"
import os
from pathlib import Path

from google.colab import drive, userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
import torch, pyannote.audio
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "| pyannote", pyannote.audio.__version__)

## Diarize

Episode audio comes from the private HF dataset (`training/episodes/`), and the declared speaker count from
its analytics export's episode metadata. Single-host episodes are diarized as one speaker, so they carry no
overlap by construction. Delete an episode's key from `diarization.json` to redo it.

In [ ]:
import json
import time
from collections import Counter

import soundfile as sf
from huggingface_hub import snapshot_download
from pyannote.audio import Pipeline

MODEL = "pyannote/speaker-diarization-community-1"
DATA = Path(snapshot_download("Sagyam/nepanglish-asr", repo_type="dataset",
                              allow_patterns=["training/episodes/*", "analytics/analytics.jsonl"]))
nspk = {}
for line in (DATA / "analytics" / "analytics.jsonl").open(encoding="utf-8"):
    r = json.loads(line)
    nspk.setdefault(r["episode_id"], len(r["episode_metadata"]["speakers"]))
print(len(nspk), "episodes; declared speakers:", dict(sorted(Counter(nspk.values()).items())))

pipe = Pipeline.from_pretrained(MODEL, token=os.environ["HF_TOKEN"])
pipe.to(torch.device("cuda"))
OUT = Path("/content/drive/MyDrive/nepanglish-asr/diarization")
OUT.mkdir(parents=True, exist_ok=True)
path = OUT / "diarization.json"
result = json.loads(path.read_text()) if path.exists() else {}  # resumes after a disconnect


def tracks(ann):
    return [[round(s.start, 3), round(s.end, 3), lab] for s, _, lab in ann.itertracks(yield_label=True)]


t_all = time.perf_counter()
for ep in sorted(nspk):
    if ep in result:
        continue
    wav, sr = sf.read(DATA / "training" / "episodes" / f"{ep}.flac", dtype="float32")
    t0 = time.perf_counter()
    out = pipe({"waveform": torch.from_numpy(wav)[None], "sample_rate": sr}, num_speakers=nspk[ep])
    dt = time.perf_counter() - t0
    emb = out.speaker_embeddings
    result[ep] = {
        "num_speakers": nspk[ep], "seconds": len(wav) / sr,
        "turns": tracks(out.speaker_diarization),              # overlapping: >1 speaker at once is kept
        "exclusive": tracks(out.exclusive_speaker_diarization),  # one speaker at a time
        "labels": out.speaker_diarization.labels(),
        "embeddings": None if emb is None else [[round(float(x), 5) for x in row] for row in emb],
    }
    path.write_text(json.dumps(result, ensure_ascii=False))
    ov = out.speaker_diarization.get_overlap().duration()
    print(f"{ep[:45]:45s} {len(wav) / sr / 60:5.1f} min in {dt:5.1f} s | overlap {ov:6.1f} s", flush=True)
print(f"done: {len(result)} episodes in {(time.perf_counter() - t_all) / 60:.1f} min -> {path}")